# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np

df = con.sql(f"""
SELECT ___id___, ___signal_a___, ___signal_b___, ___other_cols___, ___label___
FROM read_parquet('{TABLE}', hive_partitioning=true)
WHERE month = '2026-03' AND ___availability_flag___ IS TRUE
""").df()
print(df.shape)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
SIG1, LABEL = "___signal_a___", "___label___"
df["b1"] = pd.qcut(df[SIG1], 5, duplicates="drop")
t1 = df.groupby("b1", observed=True)[LABEL].agg(n="size", rate="mean").round(3)
t1

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
SIG2 = "___signal_b___"
df["b2"] = pd.qcut(df[SIG2], 5, duplicates="drop")
t2 = df.groupby("b2", observed=True)[LABEL].agg(n="size", rate="mean").round(3)
t2

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 1) Signal checks

**Signal 1: ___signal_a___ (linked to the FlyRank ___ flag, e.g. staleness / CTR-vs-position / volume).**
Verdict: **CONFIRMED / OPPOSITE / MIXED / FALSE** (pick one)
Evidence: the rate goes from ___ in the lowest bucket to ___ in the highest (n per bucket is printed above).

**Signal 2: ___signal_b___.**
Verdict: **___**
Evidence: ___

**What this means for my rule:** ___ (if a signal came out OPPOSITE or FALSE, say how I changed the rule because of it.)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def pct(s): return s.rank(pct=True)

# If a signal is CONFIRMED in the "higher = more likely" direction, use pct(x).
# If the direction is reversed, use 1 - pct(x).
df["score"] = 0.5 * pct(df[SIG1]) + 0.5 * (1 - pct(df[SIG2]))   # adjust to your verdicts

df["reason_code"] = np.where(pct(df[SIG1]) >= pct(df[SIG2]), "___REASON_A___", "___REASON_B___")

In [ ]:
ACTION = "___action_label___"   # e.g. "Refresh content" / "Fix title and meta" / "Review"
df["action"] = ACTION

import os
os.makedirs("work/outputs", exist_ok=True)

queue = (df.sort_values("score", ascending=False)
           [["___id___", "score", "reason_code", "action"]]
           .reset_index(drop=True))
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
queue.head(10)

In [ ]:
import json
top50 = df.sort_values("score", ascending=False).head(50)
metrics = {"n_rows": int(len(df)), "precision_at_50": float(top50[LABEL].mean()), "base_rate": float(df[LABEL].mean())}
json.dump(metrics, open("work/outputs/w04_baseline_metrics.json", "w"), indent=2)
metrics

## 3) Top-10 review

| # | Page | Action | Why it's here | What would make it wrong |
|---|------|--------|---------------|--------------------------|
| 1 | ___ | ___ | ___ | ___ |
| 2 | ___ | ___ | ___ | ___ |
| 3 | ___ | ___ | ___ | ___ |
| 4 | ___ | ___ | ___ | ___ |
| 5 | ___ | ___ | ___ | ___ |
| 6 | ___ | ___ | ___ | ___ |
| 7 | ___ | ___ | ___ | ___ |
| 8 | ___ | ___ | ___ | ___ |
| 9 | ___ | ___ | ___ | ___ |
| 10 | ___ | ___ | ___ | ___ |

## 4) Weak picks

Rows ___ and ___ look doubtful because ___. This is where a model could do better than my rule.

## 5) Self-check

- [x] Two signals checked, with bucket tables, n, and a one-word verdict (one linked to a FlyRank flag)
- [x] One rule: a score, one reason code, an action label
- [x] Ranked queue written from the notebook to work/outputs/baseline_action_score.csv
- [x] Ten reviewed rows, each with "what would make it wrong"
- [x] No future-window or label-derived inputs

**Lane:** I am confirming / switching my lane to ___ (lanes lock this week).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.